# Graph Animation Examples

This notebook contains a small gallery of schedules that showcase the library API.

The examples cover:

1. Concurrent transitions with `ParallelTransition`
2. Drawing curves and fills together
3. Zero-duration, sudden plot and style changes
4. Gradual alpha and color interpolation
5. Stress and jitter effects that do not change the final scene state


In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import HTML, Markdown, display

plt.rcParams["animation.html"] = "jshtml"

for candidate in (Path.cwd().resolve(), Path.cwd().resolve().parent):
    src_dir = candidate / "src"
    if src_dir.exists():
        sys.path.insert(0, str(src_dir))
        break
else:
    raise RuntimeError("Could not locate the project's src directory.")

from visualizer import (
    Curve,
    CurveStyleTransition,
    DrawTransition,
    EraseTransition,
    FillBetweenArea,
    FillBetweenTransition,
    FillStyleTransition,
    JitterTransition,
    MoveFillBetweenTransition,
    MoveTransition,
    ParallelTransition,
    Schedule,
    StressTransition,
)


In [ ]:
x = np.linspace(0.0, 1.0, 250)
y_wave = np.abs(np.sin(x))
y_square = np.square(y_wave)
y_arch = np.clip(0.2 + 0.7 * (1.0 - np.square(2.0 * x - 1.0)), 0.0, 1.0)
y_ramp = 0.12 + 0.72 * np.power(x, 1.1)


def render_example(example: dict, fps: int = 24) -> HTML:
    fig, ax = plt.subplots(figsize=(5.5, 5.5))
    anim = example["schedule"].build_animation(ax=ax, fps=fps, title=example["title"])
    ax.grid(True, alpha=0.2)
    plt.close(fig)
    return HTML(anim.to_jshtml())


def build_parallel_draws() -> Schedule:
    return Schedule().add(
        ParallelTransition(
            (
                DrawTransition(Curve("wave_a", x, y_wave, color="#0f766e", linewidth=3.0)),
                DrawTransition(
                    Curve(
                        "arch_a",
                        x,
                        y_arch,
                        color="#2563eb",
                        linewidth=2.5,
                        linestyle="--",
                    )
                ),
            )
        ),
        duration=2.0,
    )


def build_draw_and_fill_together() -> Schedule:
    schedule = Schedule()
    schedule.add(
        ParallelTransition(
            (
                DrawTransition(Curve("wave_b", x, y_wave, color="#7c3aed", linewidth=3.0)),
                FillBetweenTransition(
                    FillBetweenArea(
                        "wave_b_fill",
                        x,
                        y_wave,
                        0.0,
                        color="#c4b5fd",
                        alpha=0.25,
                        linewidth=1.0,
                    )
                ),
            )
        ),
        duration=2.0,
    )
    schedule.add(StressTransition("wave_b", color="#a78bfa", max_alpha=0.25), duration=0.8)
    return schedule


def build_sudden_plot_and_style_change() -> Schedule:
    schedule = Schedule()
    schedule.add(
        DrawTransition(
            Curve("snap", x, y_arch, color="#0ea5e9", linewidth=2.0),
            show_pointer=False,
        ),
        duration=0.0,
    )
    schedule.add(
        CurveStyleTransition("snap", color="#dc2626", linestyle="--", linewidth=3.0),
        duration=0.0,
    )
    schedule.add(StressTransition("snap", color="#fb923c", max_alpha=0.25), duration=0.8)
    return schedule


def build_fade_and_color_shift() -> Schedule:
    schedule = Schedule()
    schedule.add(
        DrawTransition(
            Curve("fade", x, y_ramp, color="#94a3b8", alpha=0.0, linewidth=3.0),
            show_pointer=False,
        ),
        duration=0.0,
    )
    schedule.add(CurveStyleTransition("fade", color="#0f766e", alpha=1.0), duration=1.5)
    schedule.add(CurveStyleTransition("fade", color="#f59e0b", alpha=0.15), duration=1.0)
    return schedule


def build_full_story() -> Schedule:
    schedule = Schedule()
    schedule.add(
        ParallelTransition(
            (
                DrawTransition(
                    Curve("story_wave", x, y_wave, color="#0f766e", linewidth=3.0)
                ),
                FillBetweenTransition(
                    FillBetweenArea(
                        "story_fill",
                        x,
                        y_wave,
                        0.0,
                        color="#99f6e4",
                        alpha=0.2,
                    )
                ),
            )
        ),
        duration=2.0,
    )
    schedule.add(StressTransition("story_wave", color="#14b8a6", max_alpha=0.3), duration=0.8)
    schedule.add(JitterTransition("story_wave", y_amplitude=0.03, cycles=12.0, seed=7), duration=0.8)
    schedule.add(
        ParallelTransition(
            (
                MoveTransition("story_wave", x_prime=x, y_prime=y_square),
                MoveFillBetweenTransition(
                    "story_fill",
                    x_prime=x,
                    y1_prime=y_square,
                    y2_prime=0.0,
                ),
                CurveStyleTransition("story_wave", color="#7c3aed"),
                FillStyleTransition("story_fill", color="#c4b5fd", alpha=0.12),
            )
        ),
        duration=2.0,
    )
    schedule.add(EraseTransition("story_wave"), duration=1.2)
    return schedule


In [ ]:
examples = [
    {
        "name": "parallel_draws",
        "title": "Parallel Draws",
        "description": "Two independent curves are drawn at the same time.",
        "schedule": build_parallel_draws(),
    },
    {
        "name": "draw_and_fill_together",
        "title": "Draw And Fill Together",
        "description": "A line and its fill_between region are revealed concurrently.",
        "schedule": build_draw_and_fill_together(),
    },
    {
        "name": "sudden_plot_and_style_change",
        "title": "Sudden Plot And Style Change",
        "description": "Zero-duration transitions produce instant plot and instant style updates.",
        "schedule": build_sudden_plot_and_style_change(),
    },
    {
        "name": "fade_and_color_shift",
        "title": "Fade And Color Shift",
        "description": "A curve is added instantly at alpha 0, then fades in and changes color.",
        "schedule": build_fade_and_color_shift(),
    },
    {
        "name": "full_story",
        "title": "Parallel Move With Effects",
        "description": "Concurrent draw/fill, stress, jitter, then move the curve and fill together.",
        "schedule": build_full_story(),
    },
]

examples


In [ ]:
[(index, example["name"], example["description"]) for index, example in enumerate(examples)]


In [ ]:
selected_index = 0
example = examples[selected_index]
display(Markdown(f"## {example['title']}"))
display(Markdown(example["description"]))
render_example(example)


In [ ]:
for example in examples:
    display(Markdown(f"## {example['title']}"))
    display(Markdown(example["description"]))
    display(render_example(example))
